In [1]:
spark.sql("DROP TABLE IF EXISTS demo.smoke.wide")
spark.sql("""
  CREATE TABLE demo.smoke.wide USING iceberg AS
  SELECT id,
         id * 2             AS a,
         id * 3             AS b,
         cast(id AS string) AS label,
         rand()             AS noise
  FROM range(1000000)
""")
spark.sql("SELECT count(*) FROM demo.smoke.wide").show()


+--------+
|count(1)|
+--------+
| 1000000|
+--------+



In [2]:
spark.sql("SELECT sum(a) FROM demo.smoke.wide").explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- BatchScan demo.smoke.wide (1)


(1) BatchScan demo.smoke.wide
Output [1]: [a#29L]
demo.smoke.wide (branch=null) [filters=, groupedBy=]

(2) HashAggregate
Input [1]: [a#29L]
Keys: []
Functions [1]: [partial_sum(a#29L)]
Aggregate Attributes [1]: [sum#42L]
Results [1]: [sum#43L]

(3) Exchange
Input [1]: [sum#43L]
Arguments: SinglePartition, ENSURE_REQUIREMENTS, [plan_id=61]

(4) HashAggregate
Input [1]: [sum#43L]
Keys: []
Functions [1]: [sum(a#29L)]
Aggregate Attributes [1]: [sum(a#29L)#33L]
Results [1]: [sum(a#29L)#33L AS sum(a)#34L]

(5) AdaptiveSparkPlan
Output [1]: [sum(a)#34L]
Arguments: isFinalPlan=false




In [3]:
spark.sql("SELECT * FROM demo.smoke.wide").explain("formatted")


== Physical Plan ==
* ColumnarToRow (2)
+- BatchScan demo.smoke.wide (1)


(1) BatchScan demo.smoke.wide
Output [5]: [id#44L, a#45L, b#46L, label#47, noise#48]
demo.smoke.wide (branch=null) [filters=, groupedBy=]

(2) ColumnarToRow [codegen id : 1]
Input [5]: [id#44L, a#45L, b#46L, label#47, noise#48]




In [6]:
spark.sql("SELECT label FROM demo.smoke.wide WHERE id = 42").explain("formatted")


== Physical Plan ==
* Project (4)
+- * Filter (3)
   +- * ColumnarToRow (2)
      +- BatchScan demo.smoke.wide (1)


(1) BatchScan demo.smoke.wide
Output [2]: [id#90L, label#93]
demo.smoke.wide (branch=null) [filters=id IS NOT NULL, id = 42, groupedBy=]

(2) ColumnarToRow [codegen id : 1]
Input [2]: [id#90L, label#93]

(3) Filter [codegen id : 1]
Input [2]: [id#90L, label#93]
Condition : (isnotnull(id#90L) AND (id#90L = 42))

(4) Project [codegen id : 1]
Output [1]: [label#93]
Input [2]: [id#90L, label#93]




In [8]:
spark.sql("SELECT sum(a) FROM demo.smoke.wide").collect()      # one column



[Row(sum(a)=999999000000)]

In [10]:

spark.sql("SELECT sum(a)+sum(b)+sum(id) FROM demo.smoke.wide").collect()  # three columns


[Row(((sum(a) + sum(b)) + sum(id))=2999997000000)]

In [11]:
spark.sql("SELECT file_path, column_sizes FROM demo.smoke.wide.files").show(truncate=False)


+-------------------------------------------------------------------------------------------+-----------------------------------------------------------------+
|file_path                                                                                  |column_sizes                                                     |
+-------------------------------------------------------------------------------------------+-----------------------------------------------------------------+
|s3://warehouse/smoke/wide/data/00000-0-b6d6d51a-777d-4360-9289-f5990b440f35-0-00001.parquet|{1 -> 188065, 2 -> 164334, 3 -> 180598, 4 -> 80299, 5 -> 1252020}|
|s3://warehouse/smoke/wide/data/00001-1-b6d6d51a-777d-4360-9289-f5990b440f35-0-00001.parquet|{1 -> 169952, 2 -> 153484, 3 -> 173589, 4 -> 90485, 5 -> 1252072}|
|s3://warehouse/smoke/wide/data/00002-2-b6d6d51a-777d-4360-9289-f5990b440f35-0-00001.parquet|{1 -> 169944, 2 -> 153483, 3 -> 173584, 4 -> 78856, 5 -> 1252007}|
|s3://warehouse/smoke/wide/data/00003-3-

In [12]:
spark.sql("""
  SELECT field_id, sum(bytes) AS total_bytes
  FROM (SELECT explode(column_sizes) AS (field_id, bytes)
        FROM demo.smoke.wide.files)
  GROUP BY field_id ORDER BY field_id
""").show()


+--------+-----------+
|field_id|total_bytes|
+--------+-----------+
|       1|    1037809|
|       2|     931757|
|       3|    1048562|
|       4|     501375|
|       5|    7512508|
+--------+-----------+



In [13]:
spark.sql("SELECT sum(a) FROM demo.smoke.wide").collect()                 # 1 column
spark.sql("SELECT sum(id)+sum(a)+sum(b) FROM demo.smoke.wide").collect()  # 3 columns


[Row(((sum(id) + sum(a)) + sum(b))=2999997000000)]